In [ ]:
import numpy as np
import pandas as pd

import zarr
import tifffile

from pathlib import Path

import gseapy as gp
import scanpy
import novae

import loki

In [ ]:
gene_list_dir = "/nfs/team361/dj16/data/MSigDB/"
path_to_gmt = Path(gene_list_dir, "HSIAO_HOUSEKEEPING_GENES.v2025.1.Hs.gmt")
house_keeping_genes_gmt = gp.parser.read_gmt(str(path_to_gmt))

df_house_keeping_genes = pd.DataFrame(house_keeping_genes_gmt).rename(columns={'HSIAO_HOUSEKEEPING_GENES':'genesymbol'})

df_house_keeping_genes.to_csv(Path(gene_list_dir, "housekeeping_genes.csv"))

In [ ]:
# tissue_type = "lung_cancer"
# data_dir = "/nfs/team361/dj16/data/Xenium_Prime_Human_Lung_Cancer_FFPE/"
# image_file = Path(data_dir, "Xenium_Prime_Human_Lung_Cancer_FFPE_he_image.ome.tif")
# transform_for_alignment = np.loadtxt(Path(data_dir, "Xenium_Prime_Human_Lung_Cancer_FFPE_he_imagealignment.csv"), delimiter=",")
# spatial_data = scanpy.read_10x_h5(Path(data_dir, "cell_feature_matrix.h5"))
# cells_df = pd.read_csv(Path(data_dir,"cells.csv.gz"))


# tissue_type = "breast_cancer"
# data_dir = "/nfs/team361/dj16/data/Xenium_Prime_Breast_Cancer_FFPE/"
# image_file = Path(data_dir, "Xenium_Prime_Breast_Cancer_FFPE_he_image.ome.tif")
# transform_for_alignment = np.loadtxt(Path(data_dir, "Xenium_Prime_Breast_Cancer_FFPE_he_imagealignment.csv"), delimiter=",")
# spatial_data = scanpy.read_10x_h5(Path(data_dir, "cell_feature_matrix.h5"))
# cells_df = pd.read_csv(Path(data_dir, "cells.csv.gz"))


# tissue_type = "ovarian_cancer"
# data_dir = "/nfs/team361/dj16/data/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun/"
# image_file = Path(data_dir, "Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_he_image.ome.tif")
# transform_for_alignment = np.loadtxt(Path(data_dir, "Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_he_imagealignment.csv"), delimiter=",")
# spatial_data = scanpy.read_10x_h5(Path(data_dir, "cell_feature_matrix.h5"))
# cells_df = pd.read_csv(Path(data_dir, "cells.csv.gz"))
# celltype_file = Path(data_dir, "Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_cell_groups.csv")


tissue_type = "lymph_node"
data_dir = "/nfs/team361/dj16/data/Xenium_Prime_Human_Lymph_Node_Reactive_FFPE/"
image_file = Path(data_dir, "Xenium_Prime_Human_Lymph_Node_Reactive_FFPE_he_image.ome.tif")
transform_for_alignment = np.loadtxt(Path(data_dir, "Xenium_Prime_Human_Lymph_Node_Reactive_FFPE_he_imagealignment.csv"), delimiter=",")
spatial_data = scanpy.read_10x_h5(Path(data_dir, "cell_feature_matrix.h5"))
cells_df = pd.read_csv(Path(data_dir, "cells.csv.gz"))
celltype_file = Path(data_dir, "Xenium_Prime_Human_Lymph_Node_Reactive_FFPE_cell_types.csv")

# Align H&E image

In [ ]:
# ===== Load image data ===== 
image_data = tifffile.imread(image_file)

# ref: https://kb.10xgenomics.com/hc/en-us/articles/35386990499853-How-can-I-convert-coordinates-between-H-E-image-and-Xenium-data?utm_source=chatgpt.com

transform_inv = np.linalg.inv(transform_for_alignment)

# ===== Load spatial data ===== 
# Extract x, y
x = cells_df["x_centroid"] / 0.2125
y = cells_df["y_centroid"] / 0.2125

# Make homogeneous coordinates: shape (3, N)
xy_hom = np.vstack([x.to_numpy(), y.to_numpy(), np.ones_like(x)])

# Apply affine transformation
xy_aligned = transform_inv @ xy_hom

# Save results back to DataFrame
cells_df["x_pixel"] = xy_aligned[1]
cells_df["y_pixel"] = xy_aligned[0]

cells_df.set_index("cell_id", inplace=True)

spatial_data.obs = cells_df.loc[spatial_data.obs_names].copy()

spatial_data.obsm["spatial"] = spatial_data.obs[["x_pixel", "y_pixel"]].to_numpy()

# Add novae embedding

In [ ]:
x1 = spatial_data

# Spatial encoder
spatial_encoder = novae.Novae.from_pretrained("MICS-Lab/novae-human-0")

# spatial embedding
novae.spatial_neighbors(x1, radius=80)
spatial_encoder.compute_representations(
    x1, zero_shot=True
)  # x should be adata

# Add NEMO embedding
- checkpoint: /nfs/team361/sb75/nichejepa-reproducibility/artifacts/hst_corpus_80m/11062025_183320_055/nichejepa-ep10_0.pth.tar

In [ ]:
# TODO

# Add cell type compostion

In [ ]:
cell_types = pd.read_csv(celltype_file, index_col=0, names=['cell_types'])

x1.obs = x1.obs.join(cell_types)

# Save anndata

In [ ]:
scanpy.write(Path(data_dir, f"spatial_preprocessed_{tissue_type}.h5"), x1)

# Tiff to zarr

In [ ]:
## image processing
image = tifffile.imread(image_file)
z = zarr.open(Path(data_dir,'image.zarr'), mode='w', shape=image.shape, chunks=(256, 256, 3), dtype='uint8')
z[:] = image
## indices processing
# 이미지 크기
H, W, C = image.shape
# 패치 크기 및 스트라이드
patch_size = 256
stride = 256
# 시작점 계산 (top-left corner)
y_coords = np.arange(0, H - patch_size + 1, stride)
x_coords = np.arange(0, W - patch_size + 1, stride)

# 모든 (y, x) 시작점 조합 생성
patch_indices = np.array([(y, x) for y in y_coords for x in x_coords], dtype=np.int32)  # shape (N, 2)
patch_indices = patch_indices[:, :, np.newaxis] # shape: (N, 2, 1)

# Zarr로 저장
zarr.save(Path(data_dir, 'indices.zarr'), patch_indices)